# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-667928b9-b53e-41f7-aaf3-ce9efe2e96f7;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [2]:
# Les chemins correspondent aux noms produits par la boucle Bronze du Jour 1.
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/olist_orders_dataset",
    "items": "/lakehouse/bronze/olist_order_items_dataset",
    "products": "/lakehouse/bronze/olist_products_dataset",
    "reviews": "/lakehouse/bronze/olist_order_reviews_dataset",
}

bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
df_bronze = bronze_dfs["main"]

key_columns = {
    "main": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ],
    "items": ["order_id", "order_item_id", "product_id", "price", "freight_value"],
    "products": ["product_id", "product_category_name", "product_weight_g"],
    "reviews": ["order_id", "review_id", "review_score", "review_comment_message"],
}
null_rows = []
for alias, df in bronze_dfs.items():
    columns = key_columns[alias]
    profile = df.agg(
        F.count("*").alias("total"),
        *[F.count(F.when(F.col(column).isNull(), 1)).alias(column) for column in columns],
    ).first()
    print(f"[{alias}] {BRONZE_TABLES[alias]} → {profile.total:,} lignes")
    null_rows.extend((alias, column, profile[column]) for column in columns)

null_report = spark.createDataFrame(null_rows, "table string, colonne string, nb_null long")
print("Valeurs NULL sur les colonnes clés")
null_report.orderBy("table", "colonne").show(len(null_rows), truncate=False)

# Les clés composites identifient un article ou un avis à l'intérieur d'une commande.
# Un order_id répété dans items ou reviews signale une relation un-à-plusieurs.
duplicate_keys = {
    "main": [("order_id",)],
    "items": [("order_id", "order_item_id"), ("order_id",)],
    "products": [("product_id",)],
    "reviews": [("order_id", "review_id"), ("order_id",)],
}
duplicate_rows = []
for alias, keys in duplicate_keys.items():
    for key in keys:
        groups = bronze_dfs[alias].dropna(subset=list(key)).groupBy(*key).count()
        summary = groups.where(F.col("count") > 1).agg(
            F.count("*").alias("groupes"),
            F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("lignes"),
        ).first()
        duplicate_rows.append((alias, " + ".join(key), summary.groupes, summary.lignes))

duplicate_report = spark.createDataFrame(
    duplicate_rows, "table string, cle string, groupes_repetes long, lignes_supplementaires long"
)
print("Clés répétées (clés NULL exclues)")
duplicate_report.orderBy("table", "cle").show(len(duplicate_rows), truncate=False)

26/11/17 11:12:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

[main] /lakehouse/bronze/olist_orders_dataset → 99,441 lignes


[items] /lakehouse/bronze/olist_order_items_dataset → 112,650 lignes


[products] /lakehouse/bronze/olist_products_dataset → 32,951 lignes


[reviews] /lakehouse/bronze/olist_order_reviews_dataset → 99,224 lignes
Valeurs NULL sur les colonnes clés


+--------+-----------------------------+-------+
|table   |colonne                      |nb_null|
+--------+-----------------------------+-------+
|items   |freight_value                |0      |
|items   |order_id                     |0      |
|items   |order_item_id                |0      |
|items   |price                        |0      |
|items   |product_id                   |0      |
|main    |customer_id                  |0      |
|main    |order_delivered_customer_date|2965   |
|main    |order_estimated_delivery_date|0      |
|main    |order_id                     |0      |
|main    |order_purchase_timestamp     |0      |
|main    |order_status                 |0      |
|products|product_category_name        |610    |
|products|product_id                   |0      |
|products|product_weight_g             |2      |
|reviews |order_id                     |0      |
|reviews |review_comment_message       |58247  |
|reviews |review_id                    |0      |
|reviews |review_sco

## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/..."  # TODO : Définir le chemin cible de votre table Silver enrichie

# TODO : 1) Nettoyer la table principale (`count_before_join`), 2) Pré-agréger/dédoublonner la source secondaire sur la clé,
#        3) Joindre les sources dans `df_silver` et vérifier l'absence d'explosion N-M
count_before_join = None
df_silver = None

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé et joint `df_silver`."
df_silver.explain("simple")

# TODO : Sauvegarder `df_silver` au format "delta" (mode overwrite + overwriteSchema=true) dans `SILVER_MAIN_PATH`
...

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
if count_before_join is not None:
    assert silver_count <= count_before_join, (
        f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
    )
print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes")

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
# TODO : Exécuter un MERGE INTO sur `SILVER_MAIN_PATH` puis comparer versionAsOf=0 et la version courante
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
...

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"